# 05 — Reproduction des figures du papier

Reproduit les analyses de la section **Usage Notes** (Nsumba et al., 2026) :

| Figure du papier | Contenu |
|---|---|
| Fig. 8 | Carte SPL moyen (Entebbe, avr–mai 2023) |
| Fig. 9 | Hourly summary: median + IQR over 24 h |
| Fig. 10 | Comparaison jour vs nuit |
| Discussion | Morphology vs SPL correlation (already done in notebook 04) |

Once the Hanoi measurements exist, these same cells produce the Hanoi figures
for the side-by-side comparison in the paper.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import HeatMap

df = pd.read_csv('../data/processed/uganda/sunbird_clean.csv')
# format='ISO8601': Sunbird timestamps mix with and without microseconds
df['timestamp'] = pd.to_datetime(df['timestamp'], format='ISO8601')
df['hour'] = df['timestamp'].dt.hour
# Standard WHO day/night definition: night = 22:00-06:00
df['period'] = np.where((df['hour'] >= 22) | (df['hour'] < 6), 'Night', 'Day')
print(f'{len(df)} mesures')

## Fig. 8 — Carte SPL moyen (heatmap spatiale)

In [ ]:
# Aggregate by grid cell (~100 m) then heatmap of mean SPL
GRID = 0.001  # ~100 m
df['lat_bin'] = (df['latitude'] / GRID).round() * GRID
df['lon_bin'] = (df['longitude'] / GRID).round() * GRID
cells = df.groupby(['lat_bin', 'lon_bin'])['noise_measurement'].mean().reset_index()

m = folium.Map(location=[df.latitude.mean(), df.longitude.mean()],
               zoom_start=12, tiles='CartoDB positron')
HeatMap(
    data=cells[['lat_bin', 'lon_bin', 'noise_measurement']].values.tolist(),
    radius=12, blur=8, min_opacity=0.4
).add_to(m)
m.save('../results/figures/sunbird/fig8_mean_spl_map.html')
print(f'{len(cells)} cellules — carte → outputs/sunbird/fig8_mean_spl_map.html')
m

## Fig. 9 - Hourly cycle: median + IQR

The paper observes: daytime median 45-50 dB, quiet mornings, a peak around 21:00-22:00.

In [ ]:
hourly = df.groupby('hour')['noise_measurement'].agg(
    median='median',
    q25=lambda x: x.quantile(0.25),
    q75=lambda x: x.quantile(0.75),
    n='count'
)

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(hourly.index, hourly['median'], marker='o', color='steelblue', label='Median')
ax.fill_between(hourly.index, hourly['q25'], hourly['q75'],
                alpha=0.25, color='steelblue', label='IQR (25–75%)')
ax.set_xlabel('Hour of day')
ax.set_ylabel('SPL (dB)')
ax.set_xticks(range(0, 24))
ax.set_title('Cycle horaire du SPL — reproduction Fig. 9')
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig('../results/figures/sunbird/fig9_hourly_cycle.png', dpi=150)
plt.show()

print('Checks against the paper:')
print(f"  Daytime median (08:00-18:00): {df[df.hour.between(8,18)]['noise_measurement'].median():.1f} dB (paper: 45-50)")
print(f"  Heure du pic : {hourly['median'].idxmax()}h (papier : 21h-22h)")

## Fig. 10 — Jour vs Nuit

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for period, color in [('Day', 'goldenrod'), ('Night', 'navy')]:
    sub = df[df['period'] == period]['noise_measurement']
    axes[0].hist(sub, bins=35, alpha=0.55, label=f'{period} (n={len(sub)})',
                 color=color, density=True)
axes[0].set_xlabel('SPL (dB)')
axes[0].set_title('Distributions jour vs nuit')
axes[0].legend()

sns.boxplot(data=df, x='period', y='noise_measurement', ax=axes[1],
            palette={'Day': 'goldenrod', 'Night': 'navy'})
axes[1].set_title('Jour vs Nuit — reproduction Fig. 10')

plt.tight_layout()
plt.savefig('../results/figures/sunbird/fig10_day_night.png', dpi=150)
plt.show()

print(df.groupby('period')['noise_measurement'].describe().round(1))

## Checklist de reproduction

| Paper element | Notebook | Status |
|---|---|---|
| Nettoyage (doublons, GPS, dB) | 02 | fait |
| QC audio (RMS, spectre, hash) | 03 | fait |
| Morphologie urbaine (R=300m) | 04 | fait |
| Carte SPL moyen (Fig. 8) | 05 | fait |
| Cycle horaire (Fig. 9) | 05 | fait |
| Jour vs nuit (Fig. 10) | 05 | fait |
| Morphology-SPL correlation | 04 | done |
| Phone calibration (Table 1) | Hanoi field | to do - cross-calibration between the phones |
| ODK Collect collection | Hanoi field | to do - same app as the paper |

**Night note**: the paper reports little night-time data (safety). Check `n` per hour
before interpreting - the same caution applies to midnight measurements in Hanoi.